# Synthethic Dataset Generator Block

Used to make a synthethic dataset modelled on the IEEE-CIS dataset

In [ ]:
import numpy as np
import pandas as pd

# =========================
# Global configuration
# =========================
SEED = 42
np.random.seed(SEED)

N_USERS = 20000
MIN_TX_PER_USER = 50
MAX_TX_PER_USER = 200
FRAUD_RATE = 0.015   # 1.5% fraud
START_TIME = 0

# Burst fraud configuration
BURST_PROB = 0.4          # fraction of fraud events that are bursts
MIN_BURST_LEN = 3
MAX_BURST_LEN = 7
BURST_TIME_SCALE = 0.05   # very small time gaps

TXN_TYPES = ["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT"]
MERCHANT_CATS = ["groceries", "electronics", "travel", "fuel", "utilities"]
CHANNELS = ["online", "pos", "mobile"]

# =========================
# Helper samplers
# =========================
def sample_categorical(dist):
    return np.random.choice(list(dist.keys()), p=list(dist.values()))

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

# =========================
# User profile generator
# =========================
def generate_user_profile():
    mu_amt = np.random.uniform(2.5, 6.0)         # log-amount mean
    sigma_amt = np.random.uniform(0.3, 0.9)
    txn_rate = np.random.uniform(0.05, 0.5)      # transactions per unit time

    type_logits = np.random.randn(len(TXN_TYPES))
    type_dist = dict(zip(TXN_TYPES, softmax(type_logits)))

    merch_logits = np.random.randn(len(MERCHANT_CATS))
    merchant_dist = dict(zip(MERCHANT_CATS, softmax(merch_logits)))

    channel_logits = np.random.randn(len(CHANNELS))
    channel_dist = dict(zip(CHANNELS, softmax(channel_logits)))

    balance_init = np.random.uniform(2_000, 20_000)

    return {
        "mu_amt": mu_amt,
        "sigma_amt": sigma_amt,
        "txn_rate": txn_rate,
        "type_dist": type_dist,
        "merchant_dist": merchant_dist,
        "channel_dist": channel_dist,
        "balance": balance_init
    }

# =========================
# Transaction generators
# =========================
def generate_normal_txn(profile, timestamp):
    amount = np.random.lognormal(profile["mu_amt"], profile["sigma_amt"])
    txn_type = sample_categorical(profile["type_dist"])
    merchant = sample_categorical(profile["merchant_dist"])
    channel = sample_categorical(profile["channel_dist"])

    old_balance = profile["balance"]
    new_balance = max(0.0, old_balance - amount)

    profile["balance"] = new_balance

    return {
        "timestamp": timestamp,
        "amount": amount,
        "oldbalanceOrg": old_balance,
        "newbalanceOrig": new_balance,
        "txn_type": txn_type,
        "merchant_cat": merchant,
        "channel": channel,
        "isFraud": 0
    }

def generate_fraud_txn(profile, timestamp):
    fraud_mode = np.random.choice(["amount", "type", "velocity", "balance"])

    old_balance = profile["balance"]

    if fraud_mode == "amount":
        amount = np.random.lognormal(profile["mu_amt"] + 4 * profile["sigma_amt"],
                                     profile["sigma_amt"])
        txn_type = sample_categorical(profile["type_dist"])

    elif fraud_mode == "type":
        rare_types = list(set(TXN_TYPES) - set(
            sorted(profile["type_dist"], key=profile["type_dist"].get, reverse=True)[:1]
        ))
        txn_type = np.random.choice(rare_types)
        amount = np.random.lognormal(profile["mu_amt"], profile["sigma_amt"])

    elif fraud_mode == "velocity":
        amount = np.random.lognormal(profile["mu_amt"], profile["sigma_amt"])
        txn_type = sample_categorical(profile["type_dist"])

    else:  # balance inconsistency
        amount = old_balance * np.random.uniform(1.2, 2.0)
        txn_type = sample_categorical(profile["type_dist"])

    merchant = sample_categorical(profile["merchant_dist"])
    channel = sample_categorical(profile["channel_dist"])

    new_balance = max(0.0, old_balance - amount)
    profile["balance"] = new_balance

    return {
        "timestamp": timestamp,
        "amount": amount,
        "oldbalanceOrg": old_balance,
        "newbalanceOrig": new_balance,
        "txn_type": txn_type,
        "merchant_cat": merchant,
        "channel": channel,
        "isFraud": 1
    }

# =========================
# Dataset generation
# =========================
records = []
txn_id = 0

for user_id in range(N_USERS):
    profile = generate_user_profile()
    n_tx = np.random.randint(MIN_TX_PER_USER, MAX_TX_PER_USER + 1)

    t = START_TIME
    # fraud_indices = set(np.random.choice(
    #     range(n_tx),
    #     size=max(1, int(FRAUD_RATE * n_tx)),
    #     replace=False
    # ))
    fraud_positions = sorted(np.random.choice(
        range(n_tx),
        size=max(1, int(FRAUD_RATE * n_tx)),
        replace=False
    ))

    fraud_schedule = {}  # index -> burst_length (1 means single fraud)

    for idx in fraud_positions:
      if np.random.rand() < BURST_PROB:
          burst_len = np.random.randint(MIN_BURST_LEN, MAX_BURST_LEN + 1)
      else:
          burst_len = 1
      fraud_schedule[idx] = burst_len

    # for i in range(n_tx):
    #     # irregular time gaps
    #     delta_t = np.random.exponential(1.0 / profile["txn_rate"])
    #     t += delta_t

    #     if i in fraud_positions:
    #         txn = generate_fraud_txn(profile, t)
    #     else:
    #         txn = generate_normal_txn(profile, t)

    #     txn["txn_id"] = txn_id
    #     txn["user_id"] = f"user_{user_id}"
    #     records.append(txn)
    #     txn_id += 1

    i = 0
    while i < n_tx:
        if i in fraud_schedule:
            burst_len = fraud_schedule[i]

            for _ in range(burst_len):
                # very short gaps during burst
                delta_t = np.random.exponential(BURST_TIME_SCALE)
                t += delta_t

                txn = generate_fraud_txn(profile, t)
                txn["txn_id"] = txn_id
                txn["user_id"] = f"user_{user_id}"
                records.append(txn)
                txn_id += 1

            i += burst_len

        else:
            delta_t = np.random.exponential(1.0 / profile["txn_rate"])
            t += delta_t

            txn = generate_normal_txn(profile, t)
            txn["txn_id"] = txn_id
            txn["user_id"] = f"user_{user_id}"
            records.append(txn)
            txn_id += 1

            i += 1


# =========================
# Save dataset
# =========================
df = pd.DataFrame(records)
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

df.to_csv("synthetic_behavioral_fraud_dataset.csv", index=False)

print("Generated dataset:")
print(df.shape)
print(df["isFraud"].value_counts(normalize=True))


# Preprocessing Block
Used to preprocess an inserted dataset. Supported datasets include:

  1.) IEEE-CIS Fraud Dataset
   
  2.) Synthethic Dataset

In [ ]:
# Settings
PATHS = {
    "ieee": {
        "train": "/content/train_transaction.csv",
        "zip": "/content/ieee-fraud-detection.zip",
    },
    "synth": {
        "train": "synthetic_behavioral_fraud_dataset.csv",
        "zip": None,
    }
}
# -------------------
# Configurable params
# # -------------------
DATASET = "synth"  # adjust path

In [ ]:
from google.colab import drive
import zipfile

ZIP_PATH = PATHS[DATASET]["zip"]

if ZIP_PATH is not None:

    # Mount Google Drive
    drive.mount('/content/drive/')

    # Specify the path to your zip file in Google Drive
    zip_file_path = ZIP_PATH

    # Create a ZipFile object
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # Extract all contents to a specified directory (e.g., /content/temp_data)
        zip_ref.extractall("/content")

    print("Files extracted successfully!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Feature engineering
AGG_WINDOW = [5, 10, 20]   # number of recent transactions to aggregate per user
# Model sizes
U_DIM = 128
Z_DIM = 128
SEQ_HIDDEN = 128
LSTM_LAYERS = 2
# GAN sizes (used below)
E_HIDDEN = [512, 256]
G_HIDDEN = [256, 512]   # flow z->h->h->x
E2_HIDDEN = [512, 256]
D_HIDDEN = [512, 256]
# Training
BATCH_SIZE = 256
LR_G = 1e-4
LR_D = 5e-5        # slower D to avoid overpowering G
N_EPOCHS = 12
CLIP_NORM = 5.0
LAMBDA_REC = 10.0   # decreased from 50
LAMBDA_LAT = 3.0    # increased importance of latent consistency
INSTANCE_NOISE_STD = 0.01
USE_INSTANCE_NOISE = True

# XGBoost params
XGB_PARAMS = dict(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=123
)

In [ ]:
# Preprocessing: build per-transaction samples with seq_recent and profile_stats
# Requirements: pandas, numpy, sklearn, torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
from tqdm import tqdm

TXN_PATH = PATHS[DATASET]["train"]

if DATASET == "ieee":
    IDENT_PATH = "/content/train_identity.csv"    # optional if you want to enrich
    USER_KEY = "user_key"                   # proxy user id column
    TIME_COL = "TransactionDT"           # time ordering column
    TARGET_COL = "isFraud"               # if present
    K = 10                               # number of previous tx to include (seq length)
    # choose numeric and categorical columns to use as transaction features.
    # Customize this list for your dataset.
    NUMERIC_COLS = ["TransactionAmt"]    # add other numeric engineered columns
    CAT_COLS = ["ProductCD", "card4", "card6"]  # categorical columns to embed later

elif DATASET == "synth":
    USER_KEY = "user_id"
    TIME_COL = "timestamp"
    TARGET_COL = "isFraud"
    K = 10
    NUMERIC_COLS = [
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig"
    ]

    CAT_COLS = [
        "txn_type",
        "merchant_cat",
        "channel"
    ]

# ----------------------------------------
# 1. Load transactions (and optional identity)
# ----------------------------------------
trans = pd.read_csv(TXN_PATH)
# optionally merge identity for extra features if available
if DATASET == "ieee":
    iden = pd.read_csv(IDENT_PATH)
    trans = trans.merge(iden, on="TransactionID", how="left")

# basic sanity
print("Transactions:", trans.shape)
if TARGET_COL not in trans.columns:
    print(f"Warning: target column {TARGET_COL} not found. Continuing without labels.")


if DATASET == "ieee":
    trans[USER_KEY] = (
    trans["card1"].astype(str) + "_" +
    trans["card2"].astype(str)
)
elif DATASET == "synth":
    # Convert user_id to string
    trans[USER_KEY] = trans[USER_KEY].astype(str)

unique_users = trans[USER_KEY].unique()
user_to_idx = {u: i for i, u in enumerate(unique_users)}

trans[USER_KEY] = trans[USER_KEY].map(user_to_idx).astype(np.int64)

print(f"Encoded {len(user_to_idx)} unique users.")

# ----------------------------------------
# 2. Minimal feature engineering
# ----------------------------------------
# create time delta features if possible (we use TransactionDT as relative timestamp)
# sort per user and compute time since last tx
trans = trans.sort_values([USER_KEY, TIME_COL]).reset_index(drop=True)
trans["txn_idx_user"] = trans.groupby(USER_KEY).cumcount()  # 0..n-1 per user
# time since previous transaction (per user)
trans["time_delta_prev"] = trans.groupby(USER_KEY)[TIME_COL].diff().fillna(0)

# log-transform amount (useful)
if "TransactionAmt" in trans.columns:
    trans["log_amt"] = np.log1p(trans["TransactionAmt"])
    NUMERIC_COLS += ["log_amt"]
    NUMERIC_COLS = list(dict.fromkeys(NUMERIC_COLS))  # dedupe

# encode missing user keys as string to avoid issues
# trans[USER_KEY] = trans[USER_KEY].astype(str)

# ----------------------------------------
# 3. Build rolling / cumulative profile stats per user (no future leakage)
#    We'll compute aggregates computed from prior transactions only.
# ----------------------------------------
# Helper: for each user group, iterate transactions and compute:
#   - seq_recent: list of up to K previous tx features
#   - profile_stats: aggregates of prior txs: mean, std, count, frac_online etc.

profile_records = []
seq_records = []
x_t_records = []
labels = []
user_ids = []

# For efficiency, define numeric feature columns used in per-step vectors
TXN_FEATURE_COLS = NUMERIC_COLS + CAT_COLS  # categorical kept raw for now (we'll embed later)
# For now we'll convert categorical to simple integer codes per column
for c in CAT_COLS:
    if c in trans.columns:
        trans[c] = trans[c].fillna("nan").astype(str).astype("category").cat.codes
    else:
        trans[c] = 0  # if column missing, create zeros

# Must ensure numeric cols exist
for c in NUMERIC_COLS:
    if c not in trans.columns:
        trans[c] = 0.0

# pre-group
groups = trans.groupby(USER_KEY)
time_records = []

print("Building per-transaction dataset (this may take a minute)...")
for user, g in tqdm(groups, total=len(groups)):
    # g is sorted by TIME_COL already
    features = g[TXN_FEATURE_COLS].values  # shape [n_user, feat_dim]
    times = g[TIME_COL].values
    n = len(g)
    # compute cumulative aggregates up to but excluding current tx
    # we can maintain running sums to be efficient
    sum_arr = np.zeros(features.shape[1], dtype=float)
    sumsq_arr = np.zeros(features.shape[1], dtype=float)
    count = 0

    # For each transaction index i, the "prior" data is features[:i]
    for i in range(n):
        # build seq_recent = up to K previous transaction feature vectors (most recent first)
        start = max(0, i - K)
        seq_window = features[start:i]  # previous transactions only
        # pad on the left so most recent is at the end; we will output order [K, feat_dim] where index -1 is most recent
        pad_len = K - seq_window.shape[0]
        if pad_len > 0:
            pad = np.zeros((pad_len, features.shape[1]), dtype=float)
            seq_padded = np.vstack([pad, seq_window])
        else:
            seq_padded = seq_window[-K:]
        # profile stats computed from prior transactions (0..i-1); if count==0, zeros
        if count == 0:
            profile = np.zeros(6, dtype=float)  # we'll use [mean, std, count, median, max, min] for numeric cols summary
        else:
            mean = sum_arr[:len(NUMERIC_COLS)] / count
            var = (sumsq_arr[:len(NUMERIC_COLS)] / count) - mean**2
            std = np.sqrt(np.maximum(var, 0.0))
            # simple aggregations aggregated across numeric features — choose to collapse into a fixed-length vector
            # We'll concatenate mean/std/count for each numeric feature; for simplicity compute aggregated summary across numeric cols
            mean_all = mean.mean()
            std_all = std.mean()
            cnt = count
            # median, min, max are expensive in running; approximate with placeholders or compute from slice
            prior_numeric = features[:i, :len(NUMERIC_COLS)]
            median_all = np.median(prior_numeric, axis=0).mean()
            max_all = np.max(prior_numeric, axis=0).mean()
            min_all = np.min(prior_numeric, axis=0).mean()
            profile = np.array([mean_all, std_all, cnt, median_all, max_all, min_all], dtype=float)
        # x_t = features[i] (current transaction features)
        x_t = features[i]
        # label if present
        if TARGET_COL in g.columns:
            y = g.iloc[i][TARGET_COL]
        else:
            y = 0

        seq_records.append(seq_padded.astype(np.float32))        # [K, feat_dim]
        profile_records.append(profile.astype(np.float32))       # [profile_dim]
        x_t_records.append(x_t.astype(np.float32))               # [feat_dim]
        labels.append(int(y))
        user_ids.append(user)
        time_records.append(times[i])

        # update running sums for next iteration
        # we update using numeric columns only (first len(NUMERIC_COLS) positions)
        num_feats = features[i, :len(NUMERIC_COLS)].astype(float)
        sum_arr[:len(NUMERIC_COLS)] += num_feats
        sumsq_arr[:len(NUMERIC_COLS)] += num_feats**2
        count += 1

# Convert to arrays
seq_array = np.stack(seq_records, axis=0)         # [N, K, feat_dim]
profile_array = np.stack(profile_records, axis=0) # [N, profile_dim]
x_t_array = np.stack(x_t_records, axis=0)         # [N, feat_dim]
labels = np.array(labels, dtype=np.int64)
user_ids = np.array(user_ids, dtype=np.int64)
time_array = np.array(time_records, dtype=np.float64)

print("Shapes:", seq_array.shape, profile_array.shape, x_t_array.shape, labels.shape, time_array.shape)

# ----------------------------------------
# 4. Scaling numeric inputs (fit on all x_t & seq numeric cols)
# ----------------------------------------
# We'll fit a StandardScaler on numeric parts of txn features (TransactionAmt, log_amt, etc.)
scaler = StandardScaler()
# flatten seq numeric values for fitting
num_idx = list(range(len(NUMERIC_COLS)))  # numeric positions in per-txn vector
seq_numeric = seq_array[:, :, :len(NUMERIC_COLS)].reshape(-1, len(NUMERIC_COLS))
x_numeric = x_t_array[:, :len(NUMERIC_COLS)]
scaler.fit(np.vstack([seq_numeric, x_numeric]))
# apply scaling to seq_array and x_t_array for numeric positions
seq_array_scaled = seq_array.copy()
seq_array_scaled[:, :, :len(NUMERIC_COLS)] = scaler.transform(seq_array[:, :, :len(NUMERIC_COLS)].reshape(-1, len(NUMERIC_COLS))).reshape(seq_array.shape[0], K, len(NUMERIC_COLS))
x_t_array_scaled = x_t_array.copy()
x_t_array_scaled[:, :len(NUMERIC_COLS)] = scaler.transform(x_t_array[:, :len(NUMERIC_COLS)])

# profile stats scaling
prof_scaler = StandardScaler()
prof_scaler.fit(profile_array)
profile_array_scaled = prof_scaler.transform(profile_array)

# ----------------------------------------
# 5. Save arrays or wrap in a PyTorch Dataset (see next block)
# ----------------------------------------
np.savez_compressed(f"{DATASET}_preprocessed_tx_dataset.npz",
                    seq=seq_array_scaled,
                    profile=profile_array_scaled,
                    x_t=x_t_array_scaled,
                    labels=labels,
                    user_ids=user_ids,
                    times=time_array)

print(f"Saved {DATASET}_preprocessed_tx_dataset")



Transactions: (2497405, 10)
Encoded 20000 unique users.
Building per-transaction dataset (this may take a minute)...


100%|██████████| 20000/20000 [12:13<00:00, 27.27it/s]


Shapes: (2497405, 10, 6) (2497405, 6) (2497405, 6) (2497405,) (2497405,)
Saved synth_preprocessed_tx_dataset


# Account Ganomaly

Contains the entire source code for the model, as well as evaluation metrics

## - Settings and Configuration

Provides available configurations that can be slotted into the code block from changing the config parameters.

In [ ]:
# Settings and Configuration

DATASET_PATHS = {
    "ieee": {
        "os_path":"/content/ieee_preprocessed_tx_dataset.npz",
        "results_path":"ieee_results_ablation.csv"
    },
    "synth": {
        "os_path":"/content/synth_preprocessed_tx_dataset.npz",
        "results_path":"synth_results_ablation.csv"
    }
}

ABALATION_REGIME = {
    "fully_global": {
        "use_ganomaly": False,
        "use_sequence": False,
        "use_profile": False,
        "use_embedding": False,
    },
    "short_term_only": {
        "use_ganomaly": True,
        "use_sequence": True,
        "use_profile": False,
        "use_embedding": True,
    },
    "long_term_only": {
        "use_ganomaly": True,
        "use_sequence": False,
        "use_profile": True,
        "use_embedding": True,
    },
    "implicit_personalization": {
        "use_ganomaly": True,
        "use_sequence": False,
        "use_profile": False,
        "use_embedding": False
    },
    "raw_embedding": {
        "use_ganomaly": True,
        "use_sequence": False,
        "use_profile": False,
        "use_embedding": True
    },
    "full_model": {
        "use_ganomaly": True,
        "use_sequence": True,
        "use_profile": True,
        "use_embedding": True
    }
}

#  ----- Configurable Values: -----
DATASET_NAME = "synth"
CURRENT_REGIME = "full_model"

#  ----- Configuration Checks -----
assert DATASET_NAME in DATASET_PATHS, f"Invalid dataset: {DATASET_NAME}"
assert CURRENT_REGIME in ABALATION_REGIME, f"Invalid regime: {CURRENT_REGIME}"
# for k, v in ABALATION_REGIME[CURRENT_REGIME].items():
#     globals()[k] = v
#  -----


## - Initialization

Contains definitions for Account Ganomaly, including classes for Ganomaly, Hybrid User Encoder, Fraud Model etc.

Also includes Settings and Configuration (for abalation)

In [ ]:
!pip install xgboost

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
import xgboost as xgb
import matplotlib.pyplot as plt
import datetime
# import multiprocessing
# multiprocessing.set_start_method("spawn", force=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Hyperparameters

CONFIG = ABALATION_REGIME[CURRENT_REGIME]

U_DIM = 128
Z_DIM = 128
SEQ_HIDDEN = 128
LSTM_LAYERS = 2
E_HIDDEN = [512, 256]
G_HIDDEN = [256, 512]
E2_HIDDEN = [512, 256]
D_HIDDEN = [512, 256]
BATCH_SIZE = 256
LR_G = 1e-4
LR_D = 5e-5
N_EPOCHS = 12
CLIP_NORM = 5.0
LAMBDA_REC = 10.0
LAMBDA_LAT = 3.0
INSTANCE_NOISE_STD = 0.01
USE_INSTANCE_NOISE = True
USE_GANOMALY = CONFIG["use_ganomaly"]
USE_SEQUENCE = CONFIG["use_sequence"]
USE_PROFILE = CONFIG["use_profile"]
USE_USER_EMBEDDING = CONFIG["use_embedding"]
OS_PATH_PREPROCESSED = DATASET_PATHS[DATASET_NAME]["os_path"]
RESULTS_PATH = DATASET_PATHS[DATASET_NAME]["results_path"]


Device: cuda


In [ ]:
if os.path.exists(OS_PATH_PREPROCESSED):
    data = np.load(OS_PATH_PREPROCESSED)
    seq = data['seq']
    profile = data['profile']
    x_t = data['x_t']
    all_labels = data['labels']
    user_ids = data['user_ids'] if 'user_ids' in data.files else np.arange(len(x_t))
    print('Loaded arrays: seq', seq.shape, 'profile', profile.shape, 'x_t', x_t.shape)
else:
    print(OS_PATH_PREPROCESSED + ' not found — please load your data manually')

Loaded arrays: seq (2497405, 10, 6) profile (2497405, 6) x_t (2497405, 6)


In [ ]:
class QuickDataset(Dataset):
    def __init__(self, seq, profile, x_t, labels, user_ids=None):
        self.seq = seq.astype(np.float32)
        self.profile = profile.astype(np.float32)
        self.x_t = x_t.astype(np.float32)
        self.all_labels = all_labels.astype(np.int64)
        self.user_ids = np.array(user_ids) if user_ids is not None else np.arange(len(self.x_t))
    def __len__(self):
        return self.x_t.shape[0]
    def __getitem__(self, idx):
        return {
            'seq_recent': torch.from_numpy(self.seq[idx]),
            'profile_stats': torch.from_numpy(self.profile[idx]),
            'x_t': torch.from_numpy(self.x_t[idx]),
            'label': torch.tensor(self.all_labels[idx], dtype=torch.long),
            'user_id': torch.tensor(self.user_ids[idx], dtype=torch.long)
        }
if 'x_t' in globals():
    idxs = np.arange(len(x_t)); np.random.seed(42); np.random.shuffle(idxs)
    split = int(0.9 * len(idxs))
    train_idx, val_idx = idxs[:split], idxs[split:]
    train_ds = torch.utils.data.Subset(QuickDataset(seq, profile, x_t, all_labels, user_ids), train_idx)
    val_ds   = torch.utils.data.Subset(QuickDataset(seq, profile, x_t, all_labels, user_ids), val_idx)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print('Created train_loader and val_loader')
else:
    print('Data not loaded; skip creating loaders')


Created train_loader and val_loader


In [ ]:
# ==========================
# Always compute validation labels + users
# ==========================

val_labels = []
val_users = []

for batch in val_loader:
    val_labels.extend(batch['label'].cpu().numpy())
    val_users.extend(batch['user_id'].cpu().numpy())

val_labels = np.array(val_labels)
val_users = np.array(val_users)

print("Validation set size:", len(val_labels))

Validation set size: 249741


In [ ]:
  # === Precompute user history length from TRAIN set ===
from collections import Counter

def compute_train_user_counts(loader):
    counter = Counter()
    for batch in loader:
        for u in batch['user_id']:
            counter[int(u)] += 1
    return dict(counter)

train_user_counts = compute_train_user_counts(train_loader)

history_counts = np.array(list(train_user_counts.values()))
print("Train history stats:",
      "min:", history_counts.min(),
      "median:", np.median(history_counts),
      "max:", history_counts.max())

import math
import torch

def history_gate(history_len, tau=7.0):
    """
    history_len: Tensor or scalar
    tau: controls how fast gate saturates
    returns gate in [0,1]
    """
    return torch.clamp(torch.log1p(history_len) / tau, 0.0, 1.0)

Train history stats: min: 37 median: 112.0 max: 191


In [ ]:
class UserHybridEncoder(nn.Module):
    def __init__(self, feat_dim, profile_dim,
                 u_dim=U_DIM,
                 seq_hidden=SEQ_HIDDEN,
                 lstm_layers=LSTM_LAYERS):
        super().__init__()

        # ALWAYS instantiate everything
        self.lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=seq_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=False
        )

        self.seq_proj = nn.Linear(seq_hidden, u_dim // 2)

        self.profile_mlp = nn.Sequential(
            nn.Linear(profile_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, u_dim // 2)
        )

        self.out_proj = nn.Linear(u_dim, u_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, seq, profile):

        # ----- Sequence branch -----
        if USE_SEQUENCE:
            _, (h_n, _) = self.lstm(seq)
            h = h_n[-1]
            u_seq = self.seq_proj(h)
        else:
            batch_size = seq.size(0)
            u_seq = torch.zeros(batch_size, U_DIM // 2, device=seq.device)

        # ----- Profile branch -----
        if USE_PROFILE:
            u_prof = self.profile_mlp(profile)
        else:
            batch_size = seq.size(0)
            u_prof = torch.zeros(batch_size, U_DIM // 2, device=seq.device)

        # Concatenate
        u = torch.cat([u_seq, u_prof], dim=1)
        u = self.dropout(u)
        u = self.out_proj(u)

        # Final embedding gate
        if USE_USER_EMBEDDING:
            return u
        else:
            return torch.zeros_like(u)

In [ ]:
class FiLM(nn.Module):
    def __init__(self, u_dim, num_features):
        super().__init__()
        self.fc_gamma = nn.Linear(u_dim, num_features)
        self.fc_beta  = nn.Linear(u_dim, num_features)
    def forward(self, x, u):
        gamma = self.fc_gamma(u)
        beta = self.fc_beta(u)
        return gamma * x + beta
class EncoderE(nn.Module):
    def __init__(self, x_dim, u_dim, hidden=E_HIDDEN, z_dim=Z_DIM):
        super().__init__(); in_dim = x_dim + u_dim; layers = []; prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h)); layers.append(nn.LayerNorm(h)); layers.append(nn.LeakyReLU(0.2)); prev = h
        self.net = nn.Sequential(*layers); self.fc_z = nn.Linear(prev, z_dim)
    def forward(self, x, u):
        xu = torch.cat([x, u], dim=1); h = self.net(xu); z = self.fc_z(h); return z
class GeneratorG(nn.Module):
    def __init__(self, z_dim=Z_DIM, u_dim=U_DIM, hidden=G_HIDDEN, x_dim=None):
        super().__init__(); x_dim = int(x_dim)
        self.fc_in = nn.Linear(z_dim, hidden[0]); self.film1 = FiLM(u_dim, hidden[0])
        self.fc_mid = nn.Linear(hidden[0], hidden[1]); self.film2 = FiLM(u_dim, hidden[1])
        self.fc_out = nn.Linear(hidden[1], x_dim)
    def forward(self, z, u):
        h = F.relu(self.film1(self.fc_in(z), u)); h = F.relu(self.film2(self.fc_mid(h), u)); x_hat = self.fc_out(h); return x_hat
class EncoderE2(nn.Module):
    def __init__(self, x_dim, u_dim, hidden=E2_HIDDEN, z_dim=Z_DIM):
        super().__init__(); in_dim = x_dim + u_dim; layers = []; prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h)); layers.append(nn.LayerNorm(h)); layers.append(nn.LeakyReLU(0.2)); prev = h
        self.net = nn.Sequential(*layers); self.fc_z = nn.Linear(prev, z_dim)
    def forward(self, x_hat, u):
        xu = torch.cat([x_hat, u], dim=1); h = self.net(xu); z_hat = self.fc_z(h); return z_hat
class DiscriminatorD(nn.Module):
    def __init__(self, x_dim, u_dim, hidden=D_HIDDEN):
        super().__init__(); in_dim = x_dim + u_dim; layers = []; prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h)); layers.append(nn.LayerNorm(h)); layers.append(nn.LeakyReLU(0.2)); prev = h
        layers.append(nn.Linear(prev, 1)); self.net = nn.Sequential(*layers)
    def forward(self, x, u): xu = torch.cat([x, u], dim=1); return self.net(xu).squeeze(1)
class ConditionalGANomaly(nn.Module):
    def __init__(self, x_dim, u_dim=U_DIM, z_dim=Z_DIM):
        super().__init__();
        self.E = EncoderE(x_dim, u_dim, z_dim=z_dim);
        self.G = GeneratorG(z_dim, u_dim, x_dim=x_dim);
        self.E2 = EncoderE2(x_dim, u_dim, z_dim=z_dim);
        self.D = DiscriminatorD(x_dim, u_dim)
    def forward(self, x, u):
        z = self.E(x, u);
        x_hat = self.G(z, u);
        z_hat = self.E2(x_hat, u);
        d_real = self.D(x, u);
        d_fake = self.D(x_hat.detach(), u);
        return dict(z=z, x_hat=x_hat, z_hat=z_hat, d_real=d_real, d_fake=d_fake)


In [ ]:
class FraudModel(nn.Module):
    def __init__(self, x_dim, profile_dim):
        super().__init__()

        self.use_ganomaly = USE_GANOMALY
        self.use_embedding = USE_USER_EMBEDDING

        self.user_encoder = UserHybridEncoder(
            feat_dim=x_dim,
            profile_dim=profile_dim
        )

        if self.use_ganomaly:
            self.ganomaly = ConditionalGANomaly(x_dim=x_dim)
        else:
            self.ganomaly = None

    def forward(self, seq, profile, x, hist_len=None):

        # -------------------------
        # User embedding
        # -------------------------
        u_raw = self.user_encoder(seq, profile)

        if self.use_embedding and hist_len is not None:
            g = history_gate(hist_len)
            u = g * u_raw
        else:
            u = torch.zeros_like(u_raw)

        # -------------------------
        # Global regime
        # -------------------------
        if not self.use_ganomaly:
            return None, None, None, u

        # -------------------------
        # GANomaly forward
        # -------------------------
        z = self.ganomaly.E(x, u)
        x_hat = self.ganomaly.G(z, u)
        z_hat = self.ganomaly.E2(x_hat, u)

        return x_hat, z, z_hat, u

In [ ]:
if 'x_t' in globals():
    feat_dim = x_t.shape[1]
    profile_dim = profile.shape[1]
    model = FraudModel(x_dim=feat_dim, profile_dim=profile_dim).to(device)

    if model.use_ganomaly:
        netD = model.ganomaly.D
    else:
        netD = None

    print("Instantiated FraudModel")
else:
    print('Data arrays not present; load data first')


Instantiated FraudModel


##  - Training

Contains the training loop for the model, requires previous module to be pre-run before execution.

In [ ]:
import time
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

if not model.use_ganomaly:
    print("Fully global regime — skipping GAN training.")
else:

    bce_logits = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()

    save_dir = 'models_account_ganomaly'
    os.makedirs(save_dir, exist_ok=True)

    optimizer_G = torch.optim.Adam(
        list(model.ganomaly.E.parameters()) +
        list(model.ganomaly.G.parameters()) +
        list(model.ganomaly.E2.parameters()) +
        list(model.user_encoder.parameters()),
        lr=LR_G, betas=(0.5, 0.999)
    )

    optimizer_D = torch.optim.Adam(
        model.ganomaly.D.parameters(),
        lr=LR_D, betas=(0.5, 0.999)
    )

    best_val_auc = -1.0

    for epoch in range(1, N_EPOCHS + 1):

        model.train()
        train_losses = []
        epoch_start = time.time()

        # ==========================
        # TRAIN
        # ==========================
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):

            seq_b = batch['seq_recent'].to(device)
            prof_b = batch['profile_stats'].to(device)
            x_b = batch['x_t'].to(device)

            if USE_USER_EMBEDDING:
                hist_len = torch.tensor(
                    [train_user_counts.get(int(u), 0) for u in batch['user_id']],
                    device=device, dtype=torch.float32
                ).unsqueeze(1)
            else:
                hist_len = None

            x_hat, z, z_hat, u = model(seq_b, prof_b, x_b, hist_len)

            # -------------------------
            # Discriminator
            # -------------------------
            with torch.no_grad():
                x_hat_det = x_hat.detach()

            d_real = model.ganomaly.D(x_b, u.detach())
            d_fake = model.ganomaly.D(x_hat_det, u.detach())

            loss_d = (
                bce_logits(d_real, torch.ones_like(d_real)) +
                bce_logits(d_fake, torch.zeros_like(d_fake))
            )

            optimizer_D.zero_grad()
            loss_d.backward()
            optimizer_D.step()

            # -------------------------
            # Generator
            # -------------------------
            d_fake_for_G = model.ganomaly.D(x_hat, u)

            loss_adv = bce_logits(d_fake_for_G, torch.ones_like(d_fake_for_G))
            loss_rec = torch.mean(torch.sum(torch.abs(x_b - x_hat), dim=1))
            loss_lat = mse(z, z_hat)

            loss_G = loss_adv + LAMBDA_REC * loss_rec + LAMBDA_LAT * loss_lat

            optimizer_G.zero_grad()
            loss_G.backward()
            optimizer_G.step()

            train_losses.append(loss_G.item())

        # ==========================
        # VALIDATION
        # ==========================
        model.eval()

        rec_scores = []
        rec_labels = []
        z_list = []
        zhat_list = []

        with torch.no_grad():
            for batch in val_loader:

                seq_b = batch['seq_recent'].to(device)
                prof_b = batch['profile_stats'].to(device)
                x_b = batch['x_t'].to(device)
                labels_b = batch['label'].cpu().numpy()

                if USE_USER_EMBEDDING:
                    hist_len = torch.tensor(
                        [train_user_counts.get(int(u), 0) for u in batch['user_id']],
                        device=device, dtype=torch.float32
                    ).unsqueeze(1)
                else:
                    hist_len = None

                x_hat, z_b, zhat_b, _ = model(seq_b, prof_b, x_b, hist_len)

                rec_b = torch.sum(torch.abs(x_b - x_hat), dim=1).cpu().numpy()

                rec_scores.append(rec_b)
                rec_labels.append(labels_b)
                z_list.append(z_b.cpu().numpy())
                zhat_list.append(zhat_b.cpu().numpy())

        rec_scores = np.concatenate(rec_scores)
        rec_labels = np.concatenate(rec_labels)

        z_val = np.vstack(z_list)
        zhat_val = np.vstack(zhat_list)

        z_diff = np.linalg.norm(z_val - zhat_val, axis=1)

        rs = (rec_scores - rec_scores.mean()) / (rec_scores.std() + 1e-9)
        zs = (z_diff - z_diff.mean()) / (z_diff.std() + 1e-9)

        score_comb = 0.5 * rs + 0.5 * zs
        val_auc = roc_auc_score(rec_labels, score_comb)

        print(f"Epoch {epoch} — "
              f"train_loss={np.mean(train_losses):.4f}, "
              f"val_auc={val_auc:.4f}, "
              f"time={(time.time()-epoch_start):.1f}s")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save({'model_state': model.state_dict()},
                       os.path.join(save_dir,
                       f"gan_best_epoch{epoch}_auc{val_auc:.4f}.pth")
            )
            print("Saved best checkpoint")

Epoch 1 — train_loss=3.1645, val_auc=0.6421, time=187.1s
Saved best checkpoint


Epoch 2 — train_loss=1.6495, val_auc=0.6673, time=192.4s
Saved best checkpoint


Epoch 3 — train_loss=1.4605, val_auc=0.6446, time=190.2s


Epoch 4 — train_loss=1.3556, val_auc=0.6213, time=187.5s


Epoch 5 — train_loss=1.2889, val_auc=0.6352, time=186.4s


...

Epoch 8 — train_loss=1.1700, val_auc=0.6220, time=186.0s


Epoch 9 — train_loss=1.1444, val_auc=0.6179, time=185.0s


Epoch 10 — train_loss=1.1225, val_auc=0.6377, time=185.3s


Epoch 11 — train_loss=1.1037, val_auc=0.6220, time=186.7s


Epoch 12 — train_loss=1.0876, val_auc=0.6377, time=190.2s


## - Inference & Evaluation

Gives a run down of model performance with pure User-encoded Ganomaly

In [ ]:
import glob
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

if model.use_ganomaly:

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------------------------
    # Load checkpoint
    # ---------------------------
    ckpts = sorted(glob.glob("models_account_ganomaly/gan_best_epoch*_auc*.pth"))
    if not ckpts:
        raise FileNotFoundError("No checkpoints found.")

    ckpt = ckpts[-1]
    print("Loading:", ckpt)

    state = torch.load(ckpt, map_location=device)

    model = FraudModel(x_dim=feat_dim, profile_dim=profile_dim).to(device)
    model.load_state_dict(state['model_state'])
    model.eval()

    # ---------------------------
    # Containers
    # ---------------------------
    recs = []
    infer_labels = []
    infer_users = []
    z_list = []
    zhat_list = []
    per_feat_list = []

    # ---------------------------
    # Inference loop
    # ---------------------------
    with torch.no_grad():
        for batch in val_loader:

            seq_b = batch['seq_recent'].to(device)
            prof_b = batch['profile_stats'].to(device)
            x_b = batch['x_t'].to(device)
            labels_b = batch['label'].cpu().numpy()
            user_b = batch['user_id'].cpu().numpy()

            if USE_USER_EMBEDDING:
                hist_len = torch.tensor(
                    [train_user_counts.get(int(u), 0) for u in user_b],
                    device=device, dtype=torch.float32
                ).unsqueeze(1)
            else:
                hist_len = None

            x_hat, z_b, zhat_b, _ = model(seq_b, prof_b, x_b, hist_len)

            # ---------------------------
            # Metrics
            # ---------------------------
            rec_b = torch.sum(torch.abs(x_b - x_hat), dim=1).cpu().numpy()
            per_feat_b = torch.abs(x_b - x_hat).cpu().numpy()

            recs.append(rec_b)
            infer_labels.append(labels_b)
            infer_users.append(user_b)
            z_list.append(z_b.cpu().numpy())
            zhat_list.append(zhat_b.cpu().numpy())
            per_feat_list.append(per_feat_b)

    # ---------------------------
    # Stack arrays
    # ---------------------------
    recs = np.concatenate(recs)
    infer_labels = np.concatenate(infer_labels)
    infer_users = np.concatenate(infer_users)

    z_val = np.vstack(z_list)
    zhat_val = np.vstack(zhat_list)
    per_feat = np.vstack(per_feat_list)

    zdiff = np.linalg.norm(z_val - zhat_val, axis=1)

    # ---------------------------
    # Combined anomaly score
    # ---------------------------
    rs = (recs - recs.mean()) / (recs.std() + 1e-9)
    zs = (zdiff - zdiff.mean()) / (zdiff.std() + 1e-9)
    score_comb = 0.5 * rs + 0.5 * zs

    # ---------------------------
    # Metrics
    # ---------------------------
    auc_rec = roc_auc_score(infer_labels, recs)
    auc_comb = roc_auc_score(infer_labels, score_comb)
    pr_auc = average_precision_score(infer_labels, score_comb)

    print("Final Metrics:")
    print(f"AUC (rec) = {auc_rec:.4f}")
    print(f"AUC (combined) = {auc_comb:.4f}")
    print(f"PR-AUC (combined) = {pr_auc:.4f}")

else:
    print("Global regime — skipping GAN inference.")

Loading: models_account_ganomaly/gan_best_epoch4_auc0.6154.pth


RuntimeError: Error(s) in loading state_dict for FraudModel:
	size mismatch for user_encoder.lstm.weight_ih_l0: copying a param with shape torch.Size([512, 5]) from checkpoint, the shape in current model is torch.Size([512, 6]).
	size mismatch for ganomaly.E.net.0.weight: copying a param with shape torch.Size([512, 133]) from checkpoint, the shape in current model is torch.Size([512, 134]).
	size mismatch for ganomaly.G.fc_out.weight: copying a param with shape torch.Size([5, 512]) from checkpoint, the shape in current model is torch.Size([6, 512]).
	size mismatch for ganomaly.G.fc_out.bias: copying a param with shape torch.Size([5]) from checkpoint, the shape in current model is torch.Size([6]).
	size mismatch for ganomaly.E2.net.0.weight: copying a param with shape torch.Size([512, 133]) from checkpoint, the shape in current model is torch.Size([512, 134]).
	size mismatch for ganomaly.D.net.0.weight: copying a param with shape torch.Size([512, 133]) from checkpoint, the shape in curr

In [ ]:
val_hist_len = np.array([
    train_user_counts.get(int(u), 0) for u in val_users
])

print("Validation history stats:",
      "min:", val_hist_len.min(),
      "median:", np.median(val_hist_len),
      "max:", val_hist_len.max())

In [ ]:
# Alpha sweep for combined score
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

best_alpha = 0.0

if model.use_ganomaly:
    rs = (recs - recs.mean()) / (recs.std() + 1e-9)
    zs = (zdiff - zdiff.mean()) / (zdiff.std() + 1e-9)

    best = {'alpha': None, 'auc': -1.0, 'prauc': -1.0}
    for alpha in np.linspace(0.0, 1.0, 11):
        score = alpha * rs + (1-alpha) * zs
        try:
            a = roc_auc_score(infer_labels, score)
            p = average_precision_score(infer_labels, score)
        except Exception:
            a, p = None, None
        print(f"alpha={alpha:.2f} -> AUC={a:.4f}, PR-AUC={p:.4f}")
        if a is not None and a > best['auc']:
            best.update({'alpha': alpha, 'auc': a, 'prauc': p})
    print("Best alpha by AUC:", best)
    best_alpha = best['alpha']

In [ ]:
def build_quantile_bins(train_user_counts):
    history_counts = np.array(list(train_user_counts.values()))

    q25, q50, q75 = np.percentile(history_counts, [25, 50, 75])

    bins = {
        "Q1 (lowest history)": (0, int(q25)),
        "Q2": (int(q25)+1, int(q50)),
        "Q3": (int(q50)+1, int(q75)),
        "Q4 (highest history)": (int(q75)+1, int(history_counts.max()))
    }

    return bins

history_bins = build_quantile_bins(train_user_counts)

print("History bins:")
for k,v in history_bins.items():
    print(k, ":", v)

In [ ]:
def assign_history_bin(history_len, bins):
    for name, (low, high) in bins.items():
        if low <= history_len <= high:
            return name
    return "Unknown"

In [ ]:
from sklearn.metrics import roc_auc_score
import pandas as pd

# print("labels:", len(labels))
# print("score_comb:", len(score_comb) if 'score_comb' in globals() else "None")
# print("xgb_probs:", len(xgb_probs) if 'xgb_probs' in globals() else "None")
# Get history length for each validation sample
# val_hist_len = np.array([
#     train_user_counts.get(int(u), 0) for u in users
# ])

if model.use_ganomaly:
    # Assign bins
    val_bins = np.array([
        assign_history_bin(h, history_bins)
        for h in val_hist_len
    ])

    results_by_bin = []

    for bin_name in history_bins.keys():
        mask = val_bins == bin_name

        if mask.sum() > 10:  # avoid tiny bins
            auc = roc_auc_score(val_labels[mask], score_comb[mask])
            results_by_bin.append((bin_name, mask.sum(), auc))

    df_gan_strat = pd.DataFrame(
        results_by_bin,
        columns=["history_bin", "num_samples", "gan_auc"]
    )

    print(df_gan_strat)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

if model.use_ganomaly:

    # ==========================
    # ROC Curve
    # ==========================
    fpr, tpr, _ = roc_curve(val_labels, score_comb)
    auc_comb = roc_auc_score(val_labels, score_comb)

    plt.figure()
    plt.plot(fpr, tpr)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve (AUC = {auc_comb:.4f})")
    plt.show()

    # ==========================
    # Precision-Recall Curve
    # ==========================
    prec, rec, _ = precision_recall_curve(val_labels, score_comb)
    pr_auc = average_precision_score(val_labels, score_comb)

    plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve (PR-AUC = {pr_auc:.4f})")
    plt.show()

    # ==========================
    # Dynamic Reconstruction Histogram
    # ==========================

    normal_recs = recs[val_labels == 0]
    fraud_recs = recs[val_labels == 1]
    all_recs = recs

    # Freedman–Diaconis rule for adaptive binning
    q25, q75 = np.percentile(all_recs, [25, 75])
    iqr = q75 - q25

    if iqr > 0:
        bin_width = 2 * iqr / (len(all_recs) ** (1/3))
        bins = int((all_recs.max() - all_recs.min()) / bin_width)
    else:
        bins = int(np.sqrt(len(all_recs)))

    # Clamp bins to reasonable range
    bins = max(20, min(bins, 300))

    plt.figure()
    plt.hist(normal_recs, bins=bins, alpha=0.6, density=True)
    plt.hist(fraud_recs, bins=bins, alpha=0.6, density=True)
    plt.xlabel("Reconstruction Error")
    plt.ylabel("Density")
    plt.title("Reconstruction Error Distribution")
    plt.show()

    # Log-scale version (if valid)
    if np.min(all_recs) > 0:
        plt.figure()
        plt.hist(normal_recs, bins=bins, alpha=0.6, density=True)
        plt.hist(fraud_recs, bins=bins, alpha=0.6, density=True)
        plt.xscale("log")
        plt.xlabel("Reconstruction Error (log scale)")
        plt.ylabel("Density")
        plt.title("Reconstruction Error Distribution (Log Scale)")
        plt.show()

## - XGBoost Training

Contains the XGBoost training section, where features from User-encoded Ganomaly are extracted and trained on the XGBoost head

In [ ]:
# TRAIN XGBOOST HEAD (may be heavy in memory)
import numpy as np, joblib
from sklearn.metrics import roc_auc_score
import xgboost as xgb

# collect train features from train_loader similarly to inference
def collect_features(loader):

    X_list = []
    y_list = []
    u_list = []

    with torch.no_grad():
        for batch in loader:

            seq_b = batch['seq_recent'].to(device)
            prof_b = batch['profile_stats'].to(device)
            x_b = batch['x_t'].to(device)

            labels_b = batch['label'].cpu().numpy()
            uids = batch['user_id']

            if model.use_ganomaly:

                u_raw = model.user_encoder(seq_b, prof_b)

                if USE_USER_EMBEDDING:
                    hist_len = torch.tensor(
                        [train_user_counts.get(int(u), 0) for u in uids],
                        device=device,
                        dtype=torch.float32
                    ).unsqueeze(1)
                    g = history_gate(hist_len)
                    u_b = g * u_raw
                else:
                    u_b = torch.zeros_like(u_raw)

                z_b = model.ganomaly.E(x_b, u_b)
                xhat_b = model.ganomaly.G(z_b, u_b)
                zhat_b = model.ganomaly.E2(xhat_b, u_b)

                rec_b = torch.sum(torch.abs(x_b - xhat_b), dim=1).cpu().numpy()
                zdiff_b = torch.norm(z_b - zhat_b, dim=1).cpu().numpy()
                per_feat_b = torch.abs(x_b - xhat_b).cpu().numpy()

                batch_X = np.concatenate([
                    rec_b.reshape(-1, 1),
                    zdiff_b.reshape(-1, 1),
                    per_feat_b,
                    z_b.cpu().numpy(),
                    u_b.cpu().numpy()
                ], axis=1)

            else:
                # ===== GLOBAL BASELINE =====
                batch_X = x_b.cpu().numpy()

            X_list.append(batch_X)
            y_list.append(labels_b)
            u_list.append(np.array(uids))

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    users = np.concatenate(u_list)

    return X, y, users

print("Collect train features...")
X_train, y_train, users_train = collect_features(train_loader)
print("Collect val features...")
X_val, y_val, users_val = collect_features(val_loader)

# user-level holdout in train for a validation split
unique_users = np.unique(users_train)
np.random.seed(123)
val_users_inner = np.random.choice(unique_users, size=int(0.1*len(unique_users)), replace=False)
train_mask = ~np.isin(users_train, val_users_inner)
val_mask = np.isin(users_train, val_users_inner)
X_tr_inner, y_tr_inner = X_train[train_mask], y_train[train_mask]
X_holdout, y_holdout = X_train[val_mask], y_train[val_mask]

clf = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric='auc', random_state=123)
clf.fit(X_tr_inner, y_tr_inner)
holdout_auc = roc_auc_score(y_holdout, clf.predict_proba(X_holdout)[:,1])
val_auc = roc_auc_score(y_val, clf.predict_proba(X_val)[:,1])
print("XGBoost holdout AUC (user-level):", holdout_auc)
print("XGBoost val AUC:", val_auc)
save_dir = 'models_account_ganomaly'
os.makedirs(save_dir, exist_ok=True)
joblib.dump(clf, "models_account_ganomaly/xgb_ganomaly_head.pkl")


## - XGBoost Head Evaluation

Contains the results obtained from training features within XGBoost, including stratified analysis

In [ ]:
print("XGBoost holdout AUC:", holdout_auc)
print("XGBoost val AUC:", val_auc)

In [ ]:
print(val_labels[:50])

In [ ]:
# === User-history stratified XGBoost evaluation ===
print("labels:", val_labels.shape)
print("val_hist_len:", val_hist_len.shape)

xgb_scores = clf.predict_proba(X_val)[:, 1]
print("xgb_scores:", xgb_scores.shape)
print("\n=== XGBoost stratified AUCs ===")

results_xgb_by_bin = []

for bin_name in history_bins.keys():
    mask = val_bins == bin_name

    if mask.sum() > 10:
        auc = roc_auc_score(val_labels[mask], xgb_scores[mask])
        results_xgb_by_bin.append((bin_name, mask.sum(), auc))

df_xgb_strat = pd.DataFrame(
    results_xgb_by_bin,
    columns=["history_bin", "num_samples", "xgb_auc"]
)

print(df_xgb_strat)

In [ ]:
# =============================================
# Results Visualization
# =============================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)

xgb_scores = clf.predict_proba(X_val)[:, 1]
labels = val_labels

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── 1. ROC Curve ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
fpr, tpr, _ = roc_curve(labels, xgb_scores)
auc_val = roc_auc_score(labels, xgb_scores)
ax1.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_val:.4f}')
ax1.plot([0,1],[0,1], 'k--', lw=1)
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve (XGBoost)'); ax1.legend(); ax1.grid(alpha=0.3)

# ── 2. Precision-Recall Curve ─────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
prec, rec, _ = precision_recall_curve(labels, xgb_scores)
prauc = average_precision_score(labels, xgb_scores)
ax2.plot(rec, prec, color='darkorange', lw=2, label=f'PR-AUC = {prauc:.4f}')
ax2.axhline(labels.mean(), color='gray', linestyle='--', lw=1, label='Baseline (fraud rate)')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve'); ax2.legend(); ax2.grid(alpha=0.3)

# ── 3. Score Distribution ─────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(xgb_scores[labels == 0], bins=60, alpha=0.6, color='steelblue', label='Legit', density=True)
ax3.hist(xgb_scores[labels == 1], bins=60, alpha=0.6, color='crimson', label='Fraud', density=True)
ax3.set_xlabel('Predicted Fraud Score'); ax3.set_ylabel('Density')
ax3.set_title('Score Distribution'); ax3.legend(); ax3.grid(alpha=0.3)

# ── 4. Stratified AUC by History Bin ─────────────────────
ax4 = fig.add_subplot(gs[1, 0])
if 'df_xgb_strat' in globals():
    bin_names = df_xgb_strat['history_bin'].tolist()
    bin_aucs  = df_xgb_strat['xgb_auc'].tolist()
    colors = ['#4C72B0','#55A868','#C44E52','#8172B2']
    bars = ax4.bar(range(len(bin_names)), bin_aucs,
                   color=colors[:len(bin_names)], edgecolor='white', width=0.6)
    ax4.set_xticks(range(len(bin_names)))
    ax4.set_xticklabels(bin_names, rotation=15, ha='right', fontsize=8)
    ax4.set_ylim(0.5, 1.0); ax4.set_ylabel('AUC')
    ax4.set_title('Stratified AUC by History Bin')
    ax4.axhline(auc_val, color='gray', linestyle='--', lw=1, label='Overall AUC')
    ax4.legend(); ax4.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, bin_aucs):
        ax4.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=8)
else:
    ax4.text(0.5, 0.5, 'df_xgb_strat not found', ha='center', va='center')
    ax4.set_title('Stratified AUC by History Bin')

# ── 5. Precision@K ────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ks = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
def precision_at_k(labels, scores, k):
    n = int(len(scores) * k) if 0 < k < 1 else int(k)
    idx = np.argsort(-scores)[:n]
    return labels[idx].sum() / max(1, n)
pak_vals = [precision_at_k(labels, xgb_scores, k) for k in ks]
ax5.plot([k*100 for k in ks], pak_vals, marker='o', color='mediumseagreen', lw=2)
ax5.axhline(labels.mean(), color='gray', linestyle='--', lw=1, label='Baseline')
ax5.set_xlabel('Top-K (%)'); ax5.set_ylabel('Precision@K')
ax5.set_title('Precision@K'); ax5.legend(); ax5.grid(alpha=0.3)
for k, v in zip(ks, pak_vals):
    ax5.annotate(f'{v:.2f}', (k*100, v), textcoords='offset points',
                 xytext=(0, 7), ha='center', fontsize=8)

# ── 6. Confusion Matrix (at best F1 threshold) ───────────
ax6 = fig.add_subplot(gs[1, 2])
prec_arr, rec_arr, thresh_arr = precision_recall_curve(labels, xgb_scores)
f1_arr = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
best_thresh = thresh_arr[np.argmax(f1_arr)]
preds = (xgb_scores >= best_thresh).astype(int)
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud'])
disp.plot(ax=ax6, colorbar=False, cmap='Blues')
ax6.set_title(f'Confusion Matrix\n(threshold = {best_thresh:.3f}, best F1)')

plt.suptitle('Account GANomaly — XGBoost Head Results', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('xgb_results_summary.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"\nOverall XGBoost AUC: {auc_val:.4f}  |  PR-AUC: {prauc:.4f}  |  Best-F1 threshold: {best_thresh:.4f}")

## - Further Evaluation

Includes extra evaluation metrics such as precision@k, Feature importance, caliberation curve etc.


In [ ]:
# precision@k and threshold table
import numpy as np
from sklearn.metrics import precision_score, recall_score

if model.use_ganomaly:
    def precision_at_k(val_labels, scores, k):
        # k is fraction (0-1) or an integer count
        if 0 < k < 1:
            n = int(len(scores) * k)
        else:
            n = int(k)
        idx = np.argsort(-scores)[:n]
        return val_labels[idx].sum() / max(1, n)

    score = 0.5 * rs + 0.5 * zs  # or use best alpha
    for k in [0.001, 0.005, 0.01, 0.02, 0.05]:  # top 0.1%, 0.5%, 1%, 2%, 5%
        print(f"precision@{k*100:.2f}% = {precision_at_k(val_labels, score, k):.4f}")

    # thresholds -> precision/recall pairs
    threshs = np.percentile(score, np.linspace(99.9, 50, 20))  # high to medium
    for t in threshs:
        preds = (score >= t).astype(int)
        p = precision_score(val_labels, preds, zero_division=0)
        r = recall_score(val_labels, preds)
        print(f"threshold={t:.4f} -> precision={p:.4f}, recall={r:.4f}, positives={preds.sum()}")


In [ ]:
# Feature importance (if clf exists)
import numpy as np
if model.use_ganomaly:
    if 'clf' in globals():
        imp = clf.get_booster().get_score(importance_type='gain')
        # sort and display top 20
        imp_sorted = sorted(imp.items(), key=lambda x: x[1], reverse=True)[:20]
        for k,v in imp_sorted:
            print(k, v)
    else:
        print("clf not in globals(); load xgb model or run training helper first.")


In [ ]:
from sklearn.calibration import calibration_curve
probs = clf.predict_proba(X_val)[:,1]
frac_pos, mean_pred = calibration_curve(y_val, probs, n_bins=10)
for i,(m,f) in enumerate(zip(mean_pred, frac_pos)):
    print(f"bin {i}: mean_pred={m:.3f}, frac_pos={f:.3f}")


In [ ]:
# Convert stratified results into flat dict entries
stratified_dict = {}

if 'df_xgb_strat' in globals():

    for _, row in df_xgb_strat.iterrows():
        bin_name = row["history_bin"]
        auc_value = row["xgb_auc"]
        num_samples = row["num_samples"]

        # sanitize column name
        col_name = f"xgb_auc_{bin_name.replace(' ', '_').replace('(', '').replace(')', '')}" # ("+str(num_samples)+")"

        stratified_dict[col_name] = auc_value
    print("Created stratified dictionary")

In [ ]:
# Recording Results
RESULTS_PATH = DATASET_NAME + "_results_ablation.csv"
def append_results(row_dict):
    if os.path.exists(RESULTS_PATH):
        df = pd.read_csv(RESULTS_PATH)
        df = pd.concat([df, pd.DataFrame([row_dict])], ignore_index=True)
    else:
        df = pd.DataFrame([row_dict])
    df.to_csv(RESULTS_PATH, index=False)
    print("Results saved to", RESULTS_PATH)

In [ ]:
# Prepare result row
result_row = {
    "regime": CURRENT_REGIME,
    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "use_ganomaly": model.use_ganomaly,
    "use_sequence": USE_SEQUENCE,
    "use_profile": USE_PROFILE,
    "use_embedding": USE_USER_EMBEDDING,
    "gan_val_auc": best_val_auc if model.use_ganomaly else None,
    "xgb_holdout_auc": holdout_auc,
    "xgb_val_auc": val_auc,
    "feature_dim": X_train.shape[1],
    **stratified_dict
}

append_results(result_row)
